In [15]:
import os
import re
import time
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs

In [16]:
def ensure_dir(f):
    if not os.path.exists(f):
        os.makedirs(f)

In [17]:
BASE_DIR = os.getcwd()  # current working directory
print(BASE_DIR)
MAIN_FOLDER = "Commission DG IV"
DEST_DIR = os.path.join(BASE_DIR, MAIN_FOLDER)
ensure_dir(DEST_DIR)

C:\Users\eduar\Documents\1 Programming\202510 -EC and ECJ\Webscraper


In [18]:
url_sources = {}
years = np.arange(1964,1998)
for i, ix in enumerate(years):
    url = f"http://ec.europa.eu/competition/antitrust/closed/en/{ix}.html"
    url_sources[ix] = url

In [19]:
# Check to see that everything is working the way it should be. Does the webscraper find anything?
caserefs = []   # store all CELEX numbers
success, failed = 0, 0  # counters

for j, (year, url) in enumerate(url_sources.items()):
    try:
        print(f"[{j}] Fetching {year} → {url}")
        response = requests.get(url, timeout=10)
        if response.status_code == 404:
            print(f"   ⚠️ {year}: Page not found (404), skipping.")
            failed += 1
            continue

        response.raise_for_status()  # raise for 4xx/5xx other than 404
        html = response.text
        print(f"   ✅ {year}: OK – {len(html)} bytes")

        # Extract CELEX numbers
        pattern = r'numdoc=([^"]*)">'
        celex_refs = re.findall(pattern, html)

        # Add to the master list
        caserefs.extend(celex_refs)
        year_count = len(celex_refs)
        success += 1

        print(f"   📄 Found {year_count} CELEX refs: {celex_refs}\n")

    except requests.exceptions.Timeout:
        print(f"   ⏱️ {year}: Request timed out, skipping.\n")
        failed += 1
    except requests.exceptions.RequestException as e:
        print(f"   ❌ {year}: HTTP Error: {e}, skipping.\n")
        failed += 1

    # Be polite to the server
    time.sleep(0.5)

print("✅ Finished fetching all years.")
print(f"   Total years processed: {success + failed}")
print(f"   Successful: {success}, Failed: {failed}")
print(f"   Total CELEX refs collected: {len(caserefs)}")


[0] Fetching 1964 → http://ec.europa.eu/competition/antitrust/closed/en/1964.html
   ✅ 1964: OK – 6234 bytes
   📄 Found 5 CELEX refs: ['364D0599', '364D0566', '364D0502', '364D0344', '364D0233']

[1] Fetching 1965 → http://ec.europa.eu/competition/antitrust/closed/en/1965.html
   ✅ 1965: OK – 5101 bytes
   📄 Found 3 CELEX refs: ['366D0005', '365D0426', '365D0366']

[2] Fetching 1966 → http://ec.europa.eu/competition/antitrust/closed/en/1966.html
   ✅ 1966: OK – 100540 bytes
   📄 Found 0 CELEX refs: []

[3] Fetching 1967 → http://ec.europa.eu/competition/antitrust/closed/en/1967.html
   ✅ 1967: OK – 4195 bytes
   📄 Found 1 CELEX refs: ['367D0454']

[4] Fetching 1968 → http://ec.europa.eu/competition/antitrust/closed/en/1968.html
   ✅ 1968: OK – 7914 bytes
   📄 Found 8 CELEX refs: ['368D0377', '368D0374', '368D0375', '368D0376', '368D0319', '368D0317', '368D0318', '368D0128']

[5] Fetching 1969 → http://ec.europa.eu/competition/antitrust/closed/en/1969.html
   ✅ 1969: OK – 8920 bytes
   

In [20]:
# Session setup with user agent
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; EC-Antitrust-Scraper/1.0)"
})

def _normalize_celex(s: str) -> str:
    """Remove all whitespace and line breaks but keep capitalization."""
    return re.sub(r"\s+", "", s or "")

def _to_celex_complete(celex_no: str, year: int | None = None) -> str:
    """
    Expand a 3+YY+L+NNNN CELEX (e.g., 371D0023) to 3+YYYY+L+NNNN (31971D0023).
    If already 3+YYYY+L+NNNN, return as-is.
    Infers 19xx/20xx automatically from the CELEX number.
    """
    c = _normalize_celex(celex_no)
    if not c:
        return ""

    # Already complete
    if re.fullmatch(r"3\d{4}[A-Z]\d{4}", c):
        return c

    # Two-digit year form
    m = re.fullmatch(r"3(\d{2})([A-Z])(\d{4})", c)
    if m:
        yy, letter, num = m.groups()

        # Convert to int to infer century
        yy_int = int(yy)
        if yy_int >= 50:
            yyyy = f"19{yy}"   # 1950–1999
        else:
            yyyy = f"20{yy}"   # 2000–2049

        return f"3{yyyy}{letter}{num}"

    # Unexpected form: return normalized
    return c

# ---- HELPER FUNCTION (robust CELEX via href + fallback) ----
def extract_entry_fields(year: int, source_url: str, a_tag) -> dict | None:
    """
    From an <a NAME="...">Title</a> anchor, scan forward for the next <font>
    that contains a date dd.mm.yyyy. Parse fields from that <font>.
    """
    title = a_tag.get_text(" ", strip=True)
    if not title or a_tag.get("name", "").lower() == "top":
        return None

    # Find the next <font> *after* the anchor that actually contains a date
    date_pat = re.compile(r"\b\d{2}\.\d{2}\.\d{4}\b")
    details_font = None

    candidate = a_tag.find_next("font")
    steps = 0
    while candidate and steps < 40:
        text = " ".join(candidate.stripped_strings)
        if date_pat.search(text):
            details_font = candidate
            break
        candidate = candidate.find_next("font")
        steps += 1

    if not details_font:
        return None

    details_text = " ".join(details_font.stripped_strings)

    # Date
    m_date = date_pat.search(details_text)
    date_str = m_date.group(0) if m_date else ""

    # Official Journal
    oj_str = ""
    m_oj = re.search(
        r"Official Journal\s*:\s*(.+?)(?=\s+Celex No\.|\s+\b[A-Z]{1,3}\s*/\s*\d{1,6}\b|$)",
        details_text
    )
    if m_oj:
        oj_str = m_oj.group(1).strip()

    # ---- CELEX (prefer href's numdoc=, fallback to displayed text) ----
    celex_no = ""        # normalized: no spaces, keep caps (e.g., '371D0023')
    celex_display = ""   # as displayed on page (may contain spaces/line breaks)

    celex_anchor = details_font.find('a', href=re.compile(r'CELEXnumdoc', re.I))
    if celex_anchor and celex_anchor.has_attr("href"):
        # Try to capture numdoc= value from href
        m_href = re.search(r"numdoc=([0-9A-Za-z]+)", celex_anchor["href"])
        if m_href:
            celex_no = _normalize_celex(m_href.group(1))
        # also keep display text (joined) in case you want it
        celex_display = celex_anchor.get_text(" ", strip=True)

    if not celex_no:
        # Fallback to text after "Celex No. :" and normalize
        m_celex_text = re.search(
            r"Celex No\.\s*:\s*([0-9A-Za-z\s]+?)(?=\s+-\s+|\s+\b[A-Z]{1,3}\s*/\s*\d{1,6}\b|$)",
            details_text
        )
        if m_celex_text:
            celex_display = m_celex_text.group(1).strip()
            celex_no = _normalize_celex(celex_display)

    # Compute full 10-char CELEX for downloads (31971D0023)
    celex_complete = _to_celex_complete(celex_no, year)

    # Case numbers (IV/…, allow 1–6 digits; optional spaces around '/')
    raw_cases = re.findall(r"\b([A-Z]{1,3})\s*/\s*(\d{1,6})\b", details_text)
    seen = set()
    case_numbers_list = []
    for prefix, num in raw_cases:
        normalized = f"{prefix}/{num}"
        if normalized not in seen:
            seen.add(normalized)
            case_numbers_list.append(normalized)
    case_numbers_str = "; ".join(case_numbers_list)

    # Decision type: after date, before OJ/CELEX/case codes
    decision_type = ""
    if date_str:
        post_date = details_text.split(date_str, 1)[1].strip()
        cut_points = []
        for marker in [
            r"Official Journal\s*:",
            r"Celex No\.\s*:",
            r"\b[A-Z]{1,3}\s*/\s*\d{1,6}\b"
        ]:
            m = re.search(marker, post_date)
            if m:
                cut_points.append(m.start())
        end_idx = min(cut_points) if cut_points else len(post_date)
        decision_type = post_date[:end_idx].strip(" -\u00A0").strip()

    return {
        "year": year,
        "title": title,
        "date": date_str,
        "decision_type": decision_type,
        "official_journal": oj_str,
        "celex_no": celex_no,               # e.g., '371D0023'
        "celex_complete": celex_complete,   # e.g., '31971D0023'  ← use this to download
        "celex_display": celex_display,     # e.g., '371D00 23' (provenance)
        "case_numbers": case_numbers_str,
        "source_url": source_url
    }

In [21]:
records = []

for year, url in url_sources.items():
    try:
        print(f"Fetching {year} → {url}")
        r = session.get(url, timeout=20)
        if r.status_code == 404:
            print(f"   404 for {year}, skipping.")
            continue

        r.raise_for_status()
        r.encoding = r.apparent_encoding or "ISO-8859-1"
        soup = BeautifulSoup(r.text, "html.parser")

        anchors = soup.find_all("a", attrs={"name": True})
        count_year = 0

        for a in anchors:
            rec = extract_entry_fields(year, url, a)
            if rec:
                records.append(rec)
                count_year += 1

        print(f"   Parsed {count_year} entries from {year}\n")
        time.sleep(0.2)

    except requests.Timeout:
        print(f"   Timeout for {year}, skipped.\n")
    except requests.RequestException as e:
        print(f"   HTTP error for {year}: {e}\n")

# ---- BUILD AND SAVE DATAFRAME ----
df = pd.DataFrame(records, columns=[
    "year", "title", "date", "decision_type",
    "official_journal", "celex_no", "celex_display", "celex_complete", "case_numbers", "source_url"
])

print(f"\nCollected {len(df)} total rows.")
print(df[df['year'] == 1970][['title','celex_display','celex_no']].head(10))  # quick check: shows both forms

out_csv = os.path.join(DEST_DIR, "antitrust_decisions.csv")
df.to_csv(out_csv, index=False, encoding="utf-8")
print(f"\n✅ Saved CSV → {out_csv}")

Fetching 1964 → http://ec.europa.eu/competition/antitrust/closed/en/1964.html
   Parsed 5 entries from 1964

Fetching 1965 → http://ec.europa.eu/competition/antitrust/closed/en/1965.html
   Parsed 3 entries from 1965

Fetching 1966 → http://ec.europa.eu/competition/antitrust/closed/en/1966.html
   Parsed 0 entries from 1966

Fetching 1967 → http://ec.europa.eu/competition/antitrust/closed/en/1967.html
   Parsed 1 entries from 1967

Fetching 1968 → http://ec.europa.eu/competition/antitrust/closed/en/1968.html
   Parsed 8 entries from 1968

Fetching 1969 → http://ec.europa.eu/competition/antitrust/closed/en/1969.html
   Parsed 10 entries from 1969

Fetching 1970 → http://ec.europa.eu/competition/antitrust/closed/en/1970.html
   Parsed 7 entries from 1970

Fetching 1971 → http://ec.europa.eu/competition/antitrust/closed/en/1971.html
   Parsed 18 entries from 1971

Fetching 1972 → http://ec.europa.eu/competition/antitrust/closed/en/1972.html
   Parsed 15 entries from 1972

Fetching 1973 → 

In [14]:
import os, time, json, random, pandas as pd
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# ---- CONFIG ----
CSV_PATH = os.path.join(DEST_DIR, "antitrust_decisions.csv")
OUT_DIR  = DEST_DIR
os.makedirs(OUT_DIR, exist_ok=True)

MAX_WAIT_S        = 18
SLEEP_BETWEEN_REQ = (0.6, 1.2)   # min/max seconds between page loads
MAX_RETRIES_BLOCK = 4            # when CDN block detected
BACKOFF_BASE_S    = 8            # exponential backoff base
COOKIE_JAR        = os.path.join(OUT_DIR, "eurlex_cookies.json")

# ---- INPUT ----
df = pd.read_csv(CSV_PATH)
celex_list = list(dict.fromkeys(
    df["celex_complete"].dropna().astype(str).str.replace(r"\s+","", regex=True).tolist()
))
print(f"Will check {len(celex_list)} CELEX ids…")

# ---- DRIVER ----
def make_driver(headless: bool = True, lang: str = "en-US,en;q=0.8"):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1600,1200")
    opts.add_argument("--lang=" + lang)
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36")
    # Hint Chrome about preferred languages
    opts.add_experimental_option("prefs", {"intl.accept_languages": lang})
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)
    driver.set_page_load_timeout(60)
    return driver

def load_cookies(driver):
    if os.path.exists(COOKIE_JAR):
        try:
            cookies = json.load(open(COOKIE_JAR, "r", encoding="utf-8"))
            driver.get("https://eur-lex.europa.eu/")  # domain context
            for ck in cookies:
                # Selenium needs minimal fields; ignore extras safely
                safe = {k: ck[k] for k in ck.keys() & {"name","value","domain","path","expiry","httpOnly","secure","sameSite"}}
                try:
                    driver.add_cookie(safe)
                except Exception:
                    pass
        except Exception:
            pass

def save_cookies(driver):
    try:
        cookies = driver.get_cookies()
        json.dump(cookies, open(COOKIE_JAR, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    except Exception:
        pass

def is_cdn_block(title: str, html: str) -> bool:
    t = (title or "").lower()
    if "request could not be satisfied" in t:
        return True
    h = (html or "").lower()
    # CloudFront / AWS fingerprints
    return ("request could not be satisfied" in h or
            "cloudfront" in h or
            "generated by cloudfront" in h or
            "bad request" in h and "cloudfront" in h)

def fetch_title(driver, url: str, celex: str, max_wait_s: int = MAX_WAIT_S):
    """Open URL, poll for a meaningful <title>. Returns (final_url, title, html)."""
    def good(t: str) -> bool:
        if not t:
            return False
        low = t.lower().strip()
        if low in {
            "the requested document does not exist. - eur-lex",
            "service temporarily unavailable - eur-lex",
            "object moved",
        }:
            return False
        return True

    driver.get(url)
    deadline = time.time() + max_wait_s
    title = ""
    while time.time() < deadline:
        try:
            t = (driver.title or "").strip()
            if good(t):
                title = t
                break
        except Exception:
            pass
        time.sleep(0.4)

    html = ""
    try:
        html = driver.page_source or ""
    except Exception:
        pass
    return driver.current_url, title, html

def save_html(path: str, html: str) -> int:
    with open(path, "w", encoding="utf-8", errors="ignore") as f:
        f.write(html)
    return os.path.getsize(path)

# ---- RUN ----
driver = make_driver(headless=True, lang="en-US,en;q=0.8")
load_cookies(driver)

records = []
ok_cnt = 0
skipped_cnt = 0
blocked_celex = []

try:
    N = len(celex_list)
    for i, celex in enumerate(celex_list, 1):
        print(f"[{i}/{N}] {celex}")
        saved_this_celex = False

        # Try EN first; if saved, skip DE to reduce load.
        for lang in ("EN", "DE"):
            if saved_this_celex:
                break

            # switch Chrome language hint for DE (optional, helps sometimes)
            if lang == "DE":
                # only toggle on DE tries to be polite
                pass

            url = f"https://eur-lex.europa.eu/legal-content/{lang}/TXT/HTML/?uri=CELEX:{celex}"

            retries_left = MAX_RETRIES_BLOCK
            attempt = 0
            while True:
                attempt += 1
                final_url, title, html = fetch_title(driver, url, celex)

                if title and celex in title:
                    out_name = f"{celex}_{lang}.html"
                    out_path = os.path.join(OUT_DIR, out_name)
                    if not os.path.exists(out_path):
                        size = save_html(out_path, html)
                    else:
                        size = os.path.getsize(out_path)

                    print(f"   {lang}: {final_url}\n      → title: {title}")
                    ok_cnt += 1
                    records.append({
                        "celex_complete": celex, "lang": lang, "url": final_url,
                        "title": title, "saved_path": out_path, "bytes": size, "note": "saved"
                    })
                    saved_this_celex = True
                    break

                # detect CDN block
                if is_cdn_block(title, html):
                    print(f"   {lang}: {final_url}\n      → CDN block detected (attempt {attempt})")
                    if retries_left > 0:
                        wait_s = BACKOFF_BASE_S * (2 ** (MAX_RETRIES_BLOCK - retries_left)) + random.uniform(0.5, 2.0)
                        time.sleep(wait_s)
                        retries_left -= 1
                        continue
                    else:
                        blocked_celex.append(celex)
                        records.append({
                            "celex_complete": celex, "lang": lang, "url": final_url,
                            "title": title or "CDN_BLOCK", "saved_path": "", "bytes": 0, "note": "cdn_block"
                        })
                        break

                # not a block, just no title/mismatch → record + stop trying this lang
                print(f"   {lang}: {final_url}\n      → title: {(title or '(empty)')}")
                skipped_cnt += 1
                records.append({
                    "celex_complete": celex, "lang": lang, "url": final_url,
                    "title": title, "saved_path": "", "bytes": 0, "note": "no_title_or_mismatch"
                })
                break

            # polite pacing between language attempts
            time.sleep(random.uniform(*SLEEP_BETWEEN_REQ))

        # small pacing between CELEX ids
        time.sleep(random.uniform(*SLEEP_BETWEEN_REQ))

finally:
    # persist cookies for next run
    save_cookies(driver)
    driver.quit()

print("\nDone.")
print(f"   ✅ Saved:   {ok_cnt}")
print(f"   ⚪ Skipped: {skipped_cnt}")
if blocked_celex:
    print(f"   ⛔ CDN-blocked CELEX (deferred): {len(set(blocked_celex))}")

idx = pd.DataFrame(records)
idx_path = os.path.join(OUT_DIR, "celex_html_index.csv")
idx.to_csv(idx_path, index=False)
print(f"Index saved → {idx_path}")




Will check 378 CELEX ids…
[1/378] 31964D0599
   EN: https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0599
      → title: (empty)
   DE: https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0599
      → title: EUR-Lex - 31964D0599 - DE
[2/378] 31964D0566
   EN: https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0566
      → title: (empty)
   DE: https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0566
      → title: EUR-Lex - 31964D0566 - DE
[3/378] 31964D0502
   EN: https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0502
      → title: (empty)
   DE: https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0502
      → title: EUR-Lex - 31964D0502 - DE
[4/378] 31964D0344
   EN: https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0344
      → title: (empty)
   DE: https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0344
      → title: EUR-Lex - 31964D0344

In [22]:
# ONE-CELL VERSION (HTML first, then DE-PDF fallback if no HTML was saved)

import os, time, json, random, pandas as pd, requests
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# ---- CONFIG ----
CSV_PATH = os.path.join(DEST_DIR, "antitrust_decisions.csv")  # assumes DEST_DIR is defined
OUT_DIR  = DEST_DIR
os.makedirs(OUT_DIR, exist_ok=True)

MAX_WAIT_S        = 18
SLEEP_BETWEEN_REQ = (0.6, 1.2)   # min/max seconds between page loads
MAX_RETRIES_BLOCK = 4            # when CDN block detected
BACKOFF_BASE_S    = 8            # exponential backoff base
COOKIE_JAR        = os.path.join(OUT_DIR, "eurlex_cookies.json")

# ---- INPUT ----
df = pd.read_csv(CSV_PATH)
celex_list = list(dict.fromkeys(
    df["celex_complete"].dropna().astype(str).str.replace(r"\s+","", regex=True).tolist()
))
print(f"Will check {len(celex_list)} CELEX ids…")

# ---- HELPERS ----
def make_driver(headless: bool = True, lang: str = "en-US,en;q=0.8"):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1600,1200")
    opts.add_argument("--lang=" + lang)
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36")
    opts.add_experimental_option("prefs", {"intl.accept_languages": lang})
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)
    driver.set_page_load_timeout(60)
    return driver

def load_cookies(driver):
    if os.path.exists(COOKIE_JAR):
        try:
            cookies = json.load(open(COOKIE_JAR, "r", encoding="utf-8"))
            driver.get("https://eur-lex.europa.eu/")  # domain context
            for ck in cookies:
                safe = {k: ck[k] for k in ck.keys() & {"name","value","domain","path","expiry","httpOnly","secure","sameSite"}}
                try:
                    driver.add_cookie(safe)
                except Exception:
                    pass
        except Exception:
            pass

def save_cookies(driver):
    try:
        cookies = driver.get_cookies()
        json.dump(cookies, open(COOKIE_JAR, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    except Exception:
        pass

def is_cdn_block(title: str, html: str) -> bool:
    t = (title or "").lower()
    if "request could not be satisfied" in t:
        return True
    h = (html or "").lower()
    return ("request could not be satisfied" in h or
            "cloudfront" in h or
            "generated by cloudfront" in h or
            ("bad request" in h and "cloudfront" in h))

def fetch_title(driver, url: str, celex: str, max_wait_s: int = MAX_WAIT_S):
    """Open URL, poll for a meaningful <title>. Returns (final_url, title, html)."""
    def good(t: str) -> bool:
        if not t:
            return False
        low = t.lower().strip()
        if low in {
            "the requested document does not exist. - eur-lex",
            "service temporarily unavailable - eur-lex",
            "object moved",
        }:
            return False
        return True

    driver.get(url)
    deadline = time.time() + max_wait_s
    title = ""
    while time.time() < deadline:
        try:
            t = (driver.title or "").strip()
            if good(t):
                title = t
                break
        except Exception:
            pass
        time.sleep(0.4)

    html = ""
    try:
        html = driver.page_source or ""
    except Exception:
        pass
    return driver.current_url, title, html

def save_html(path: str, html: str) -> int:
    with open(path, "w", encoding="utf-8", errors="ignore") as f:
        f.write(html)
    return os.path.getsize(path)

def requests_session_from_driver(driver, lang_hdr="de-DE,de;q=0.9,en-US;q=0.8"):
    """Build a requests.Session that carries over Selenium cookies & headers."""
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": lang_hdr,
        "Accept": "*/*",
        "Connection": "keep-alive",
    })
    try:
        driver.get("https://eur-lex.europa.eu/")
        for ck in driver.get_cookies():
            domain = ck.get("domain", "").lstrip(".") or "eur-lex.europa.eu"
            s.cookies.set(name=ck.get("name"),
                          value=ck.get("value"),
                          domain=domain,
                          path=ck.get("path", "/"))
    except Exception:
        pass
    return s

def is_pdf_bytes(data: bytes) -> bool:
    return data[:5] == b"%PDF-"

def try_download_pdf_with_cookies(driver, pdf_url: str, out_path: str,
                                  max_retries: int = 3, backoff_base: int = 6):
    """Download a PDF via requests + Selenium cookies. Returns (ok, size, note, final_url)."""
    sess = requests_session_from_driver(driver)
    last_note = "pdf_failed"
    final_url = pdf_url
    for attempt in range(1, max_retries + 1):
        try:
            r = sess.get(pdf_url, allow_redirects=True, timeout=60)
            final_url = r.url
            content = r.content or b""
            if r.status_code == 200 and is_pdf_bytes(content):
                with open(out_path, "wb") as f:
                    f.write(content)
                return True, os.path.getsize(out_path), "pdf_saved", final_url
            else:
                text_low = (content[:4096].decode("latin1", errors="ignore")).lower()
                if ("cloudfront" in text_low or
                    "request could not be satisfied" in text_low or
                    "generated by cloudfront" in text_low):
                    last_note = "pdf_cdn_block"
                else:
                    last_note = f"pdf_http_{r.status_code}"
        except Exception:
            last_note = "pdf_exception"
        time.sleep(backoff_base * (2 ** (attempt - 1)) + random.uniform(0.5, 1.5))
    return False, 0, last_note, final_url

# ---- RUN ----
driver = make_driver(headless=True, lang="en-US,en;q=0.8")
load_cookies(driver)

records = []
ok_cnt = 0
skipped_cnt = 0
blocked_celex = []

try:
    N = len(celex_list)
    for i, celex in enumerate(celex_list, 1):
        print(f"[{i}/{N}] {celex}")
        saved_this_celex = False

        # Try EN first; if saved, skip DE to reduce load. If both fail → try DE-PDF.
        for lang in ("EN", "DE"):
            if saved_this_celex:
                break

            url = f"https://eur-lex.europa.eu/legal-content/{lang}/TXT/HTML/?uri=CELEX:{celex}"

            retries_left = MAX_RETRIES_BLOCK
            attempt = 0
            while True:
                attempt += 1
                final_url, title, html = fetch_title(driver, url, celex)

                if title and celex in title:
                    out_name = f"{celex}_{lang}.html"
                    out_path = os.path.join(OUT_DIR, out_name)
                    if not os.path.exists(out_path):
                        size = save_html(out_path, html)
                    else:
                        size = os.path.getsize(out_path)

                    print(f"   {lang}: {final_url}\n      → title: {title}")
                    ok_cnt += 1
                    records.append({
                        "celex_complete": celex, "lang": lang, "url": final_url,
                        "title": title, "saved_path": out_path, "bytes": size, "note": "saved"
                    })
                    saved_this_celex = True
                    break

                # detect CDN block
                if is_cdn_block(title, html):
                    print(f"   {lang}: {final_url}\n      → CDN block detected (attempt {attempt})")
                    if retries_left > 0:
                        wait_s = BACKOFF_BASE_S * (2 ** (MAX_RETRIES_BLOCK - retries_left)) + random.uniform(0.5, 2.0)
                        time.sleep(wait_s)
                        retries_left -= 1
                        continue
                    else:
                        blocked_celex.append(celex)
                        records.append({
                            "celex_complete": celex, "lang": lang, "url": final_url,
                            "title": title or "CDN_BLOCK", "saved_path": "", "bytes": 0, "note": "cdn_block"
                        })
                        break

                # not a block, just no title/mismatch → record + stop trying this lang
                print(f"   {lang}: {final_url}\n      → title: {(title or '(empty)')}")
                skipped_cnt += 1
                records.append({
                    "celex_complete": celex, "lang": lang, "url": final_url,
                    "title": title, "saved_path": "", "bytes": 0, "note": "no_title_or_mismatch"
                })
                break

            # polite pacing between language attempts
            time.sleep(random.uniform(*SLEEP_BETWEEN_REQ))

        # If neither EN nor DE HTML was saved, try the DE PDF fallback
        if not saved_this_celex:
            pdf_url = f"https://eur-lex.europa.eu/legal-content/DE/TXT/PDF/?uri=CELEX:{celex}"
            out_name_pdf = f"{celex}_DE.pdf"
            out_path_pdf = os.path.join(OUT_DIR, out_name_pdf)

            if not os.path.exists(out_path_pdf):
                ok, size, note, final_url = try_download_pdf_with_cookies(driver, pdf_url, out_path_pdf)
                if ok:
                    print(f"   DE-PDF: {final_url}\n      → saved PDF: {out_path_pdf} ({size} bytes)")
                    ok_cnt += 1
                    records.append({
                        "celex_complete": celex, "lang": "DE", "url": final_url,
                        "title": "PDF (no title)", "saved_path": out_path_pdf, "bytes": size, "note": note
                    })
                    saved_this_celex = True
                else:
                    print(f"   DE-PDF: {pdf_url}\n      → {note}")
                    records.append({
                        "celex_complete": celex, "lang": "DE", "url": pdf_url,
                        "title": "", "saved_path": "", "bytes": 0, "note": note
                    })
            else:
                print(f"   DE-PDF: already present on disk → {out_path_pdf}")
                records.append({
                    "celex_complete": celex, "lang": "DE", "url": pdf_url,
                    "title": "PDF (cached)", "saved_path": out_path_pdf, "bytes": os.path.getsize(out_path_pdf), "note": "pdf_cached"
                })

        # small pacing between CELEX ids
        time.sleep(random.uniform(*SLEEP_BETWEEN_REQ))

finally:
    # persist cookies for next run
    save_cookies(driver)
    driver.quit()

print("\nDone.")
print(f"   ✅ Saved:   {ok_cnt}")
print(f"   ⚪ Skipped: {skipped_cnt}")
if blocked_celex:
    print(f"   ⛔ CDN-blocked CELEX (deferred): {len(set(blocked_celex))}")

idx = pd.DataFrame(records)
idx_path = os.path.join(OUT_DIR, "celex_html_index.csv")
idx.to_csv(idx_path, index=False)
print(f"Index saved → {idx_path}")


Will check 378 CELEX ids…
[1/378] 31964D0599
   EN: https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0599
      → title: (empty)
   DE: https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0599
      → title: EUR-Lex - 31964D0599 - DE
[2/378] 31964D0566
   EN: https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0566
      → title: (empty)
   DE: https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0566
      → title: EUR-Lex - 31964D0566 - DE
[3/378] 31964D0502
   EN: https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0502
      → title: (empty)
   DE: https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0502
      → title: EUR-Lex - 31964D0502 - DE
[4/378] 31964D0344
   EN: https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:31964D0344
      → title: (empty)
   DE: https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:31964D0344
      → title: EUR-Lex - 31964D0344